In [1]:
using LinearAlgebra, Polynomials, Plots
using Revise, DelimitedFiles, BenchmarkTools
using CloudAtlas, BifurcationKit, ChannelflowWrapper
using Dates
using Random
using Base.Threads

"""
    myreaddlm(filename, cc='%')

Read matrix or vector from a file, dropping comments marked with cc.
"""
function myreaddlm(filename; cc='%')
    X = readdlm(filename, comments=true, comment_char=cc)
    if size(X,2) == 1
        X = X[:,1]
    end
    X
end

macro suppress(ex)
    quote
        # Generate a unique name for the old stdout to avoid variable collision
        local old_stdout = stdout
        redirect_stdout(devnull)
        try
            # We use esc(ex) to run the expression in the caller's scope
            $(esc(ex))
        finally
            redirect_stdout(old_stdout)
        end
    end
end

sx, sy, sz, tx, tz = halfbox_symmetries()

pwd()

"/home/ebenq/dev/MyCloudAtlas.jl/notebooks/tw_fuzzing-updated"

In [2]:
# Parameters
hookparams = SearchParams(ftol=1e-08, xtol=1e-12, Nnewton=30,Nhook=8,δ=0.01, verbosity=0)
Re = 300.0
cx0 = 0.000 # values from paper
cz0 = 0.009

α, γ = 2π/4.0, 2π/6.0               # Fourier wavenumbers α, γ = 2π/Lx, 2π/Lz
normalize = true                    # Normalize the basis set or not?
H = [(sx*sy)*(tx*tz)]               # Generators of the symmetric subspace of TW1

# The DNS file to project from (ensure this path is correct)
dns_file = "TW1-2pi1piRe200-40x49x40.nc" 

# List of resolutions to test: [(J, K, L), ...]
# discretizations = [(1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7), (3, 5, 9)]
# discretizations = [(1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7)]
# discretizations = [(1, 1, 3), (1, 2, 3)]
discretizations = [(2, 4, 7), (3, 5, 9), (3, 5, 7)]

3-element Vector{Tuple{Int64, Int64, Int64}}:
 (2, 4, 7)
 (3, 5, 9)
 (3, 5, 7)

In [3]:
struct SolutionFingerprint
    cx::Float64
    cz::Float64
    nm::Float64
end

# Helper to check if a solution is effectively new
function is_distinct(new_fp::SolutionFingerprint, archive::Vector{SolutionFingerprint}; tol=1e-4)
    for fp in archive
        # If wave speeds AND norm are identical (within tolerance), it's a duplicate
        if isapprox(new_fp.cx, fp.cx, atol=tol) && 
           isapprox(new_fp.cz, fp.cz, atol=tol) && 
           isapprox(new_fp.nm, fp.nm, atol=tol)
            return false # It's a duplicate
        end
    end
    return true
end

is_distinct (generic function with 1 method)

In [4]:
function fuzz_symmetry_space(discretizations, H, Re; 
                             attempts_per_level=25, 
                             base_dir="fuzz_results",
                             symm_file = "./sxytxz.asc",
                             reference_path = "./TW1-2pi1piRe200-40x49x40.nc",
                             norm_threshold=1e-2,
                             xnorm = 0.40,
                             α=α, γ=γ, hookparams=hookparams, T=10.0)
    
    # 1. Setup Directory
    mkpath(base_dir)
    
    # --- Thread Safety Tools ---
    io_lock = ReentrantLock()        # For printing
    data_lock = ReentrantLock()      # NEW: For accessing the solution archive
    total_found = Atomic{Int}(0)
    
    # NEW: Archive to store fingerprints of found solutions
    # We store tuples of (cx, cz, norm) to quickly identify duplicates
    solution_archive = Vector{SolutionFingerprint}()

    reference_field_converted = "reference_field_$(α)_$(γ).nc"
    changegrid(reference_path, reference_field_converted; al=α, ga=γ)
    
    println("Starting Smart Fuzz Search in $base_dir with $(nthreads()) threads")

    for (J, K, L) in discretizations
        lock(io_lock) do 
            println("\n" * "="^60)
            println("  Discretization: J=$J, K=$K, L=$L")
            println("="^60)
        end

        # Pre-calculate model for this level
        model = TWModel(α, γ, J, K, L, H; normalize=false)
        m = length(model)

        @threads for i in 1:attempts_per_level
            
            # ... [Random Guess Generation code remains the same] ...
            x_guess = randn(m)
            x_guess = xnorm/norm(x_guess) * x_guess
            cx_guess = randn() * 0.1
            cz_guess = randn() * 0.1
            ξ_guess = [x_guess; cx_guess; cz_guess]

            # Define closures for solver
            f(ξ) = model.g(ξ, Re)
            Df(ξ) = model.Dg(ξ, Re)
            
            # Try Low-Dimensional Solve
            ξ_star, converged = hookstepsolve(f, Df, ξ_guess, hookparams)

            # Check convergence basics
            solution_norm = norm(ξ_star[1:m])
            
            if converged && solution_norm > norm_threshold && ξ_star[end - 1] > 1e-7
                println("found one!")
                x_found, cx_found, cz_found = extract_components(ξ_star, model)

                # Create a fingerprint for this solution
                new_fp = SolutionFingerprint(cx_found, cz_found, solution_norm)
                
                is_new = false
                lock(data_lock) do
                    if is_distinct(new_fp, solution_archive)
                        push!(solution_archive, new_fp)
                        is_new = true
                    end
                end
                
                if !is_new
                    # Skip expensive findsoln if we've seen this wave before
                    # Optional: Print a "skip" message if you want to track efficiency
                    # lock(io_lock) do println("  [Thread $(threadid())] Skipped duplicate (cx=$cx_found)") end
                    continue 
                end

                # --- Proceed to Save and Refine (Only for unique solutions) ---
                atomic_add!(total_found, 1)
                
                lock(io_lock) do
                    println("  [Thread $(threadid())] Hit! Unique Solution found (cx=$(round(cx_found, digits=5))) (cz=$(round(cz_found, digits=5)))")
                end
                
                # ... [File saving and findsoln call code remains the same] ...
                timestamp = Dates.format(now(), "MM-DD-HHMMSS")
                sol_dir = joinpath(base_dir, "sol_$(J)_$(K)_$(L)_id$(i)_$(timestamp)")
                mkpath(sol_dir)
                guess_path = joinpath(sol_dir, "u_guess.nc")
                sigma_file = joinpath(sol_dir, "sigma.asc")
                
                lock(io_lock) do
                    coeff2field(ξ_star[1:m], model.ijkl, reference_field_converted, guess_path)
                    save_sigma(model, ξ_star[end - 1], ξ_star[end], T, sigma_file)
                end
                
                try
                    findsoln(guess_path;
                        R = Re, eqb = true, xrel = model.keep_cx, zrel = model.keep_cz,
                        symms = abspath(symm_file), sigma = sigma_file, od = sol_dir, T = T
                    )
                catch e
                    lock(io_lock) do
                        println("  [Thread $(threadid())] findsoln failed: $e")
                    end
                end
            end
        end
    end
end

fuzz_symmetry_space (generic function with 1 method)

In [5]:
fuzz_symmetry_space(discretizations, H, Re; attempts_per_level=500, α=α, γ=γ, hookparams=hookparams)

Leaving rescale
L2Norm(u0)  == 0.2792590266581449
L2Norm(u1)  == 0.2792590266581449
bcNorm(u0)  == 8.737886593015271e-17
bcNorm(u1)  == 1.001456466862179e-16
divNorm(u0) == 4.746636220033689e-16
divNorm(u1) == 6.392261886532839e-16
L2Norm(u2)  == 0.2792590266581449
divNorm(u2) == 6.271116005970482e-18
bcNorm(u2)  == 4.580081077120656e-17
Starting Smart Fuzz Search in fuzz_results with 1 threads

  Discretization: J=2, K=4, L=7
J,K,L,m == 2,4,7,338
(2J+1)(2K+1)(2L+1) + 1 == 676
Making matrices B, A1, A2, Cx, Cz...
Phase constraints: keep_cx = false, keep_cz = true
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.154606
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00825909
   previous gx == 0.00825909
   current  gx == 0.00825909
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 3.41063e-05
   previous rx == 3.41063e-05
   current  rx == 3.41063e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.659447
Newt,GMRES == 0,1, f^T: ....10 res == 0.6559
Newt,GMRES == 0,2, f^T: ....10 res == 0.644405
Newt,GMRES == 0,3, f^T: ....10 res == 0.517157
Newt,GMRES == 0,4, f^T: ....10 res == 0.375848
Newt,GMRES == 0,5, f^T: ....10 res == 0.23155
Newt,GMRES == 0,6, f^T: ....10 res == 0.128462
Newt,GMRES == 0,7, f^T: ....10 res == 0.051703
Newt,GMRES == 0,8, f^T: ....10 res == 0.0247398
Newt,GMRES == 0,9, f^T: ....10 res == 0.0170039
Newt,GMRES == 0,10, f^T: ....10 res == 0.0111642
Newt,GMRE

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.154606
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00523976
   previous gx == 0.00523976
   current  gx == 0.00523976
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 1.37276e-05
   previous rx == 1.37276e-05
   current  rx == 1.37276e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.618629
Newt,GMRES == 0,1, f^T: ....10 res == 0.61619
Newt,GMRES == 0,2, f^T: ....10 res == 0.525346
Newt,GMRES == 0,3, f^T: ....10 res == 0.511867
Newt,GMRES == 0,4, f^T: ....10 res == 0.495483
Newt,GMRES == 0,5, f^T: ....10 res == 0.464856
Newt,GMRES == 0,6, f^T: ....10 res == 0.425515
Newt,GMRES == 0,7, f^T: ....10 res == 0.395959
Newt,GMRES == 0,8, f^T: ....10 res == 0.370144
Newt,GMRES == 0,9, f^T: ....10 res == 0.262079
Newt,GMRES == 0,10, f^T: ....10 res == 0.209412
Newt,GMRES

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0608293
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00321878
   previous gx == 0.00321878
   current  gx == 0.00321878
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 5.18028e-06
   previous rx == 5.18028e-06
   current  rx == 5.18028e-06
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.592275
Newt,GMRES == 0,1, f^T: ....10 res == 0.590509
Newt,GMRES == 0,2, f^T: ....10 res == 0.311513
Newt,GMRES == 0,3, f^T: ....10 res == 0.250774
Newt,GMRES == 0,4, f^T: ....10 res == 0.126753
Newt,GMRES == 0,5, f^T: ....10 res == 0.11196
Newt,GMRES == 0,6, f^T: ....10 res == 0.0987063
Newt,GMRES == 0,7, f^T: ....10 res == 0.075786
Newt,GMRES == 0,8, f^T: ....10 res == 0.0374925
Newt,GMRES == 0,9, f^T: ....10 res == 0.0352646
Newt,GMRES == 0,10, f^T: ....10 res == 0.0158008
Newt,

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0766028
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00439497
   previous gx == 0.00439497
   current  gx == 0.00439497
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 9.65787e-06
   previous rx == 9.65787e-06
   current  rx == 9.65787e-06
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.567096
Newt,GMRES == 0,1, f^T: ....10 res == 0.554998
Newt,GMRES == 0,2, f^T: ....10 res == 0.278123
Newt,GMRES == 0,3, f^T: ....10 res == 0.175784
Newt,GMRES == 0,4, f^T: ....10 res == 0.0398991
Newt,GMRES == 0,5, f^T: ....10 res == 0.00412575
Newt,GMRES == 0,6, f^T: ....10 res == 0.000518472
GMRES converged. Breaking.
------------------------------------------------
Beginning hookstep calculations.
-------------------------------------------
Hookstep number 0
delta == 0.01
Newton